# Filtrado colaborativo usuario - usuario basado en K-NN

Predice la valoración de un libro para un usuario a partir de los usuarios más similares. Aquí se utiliza KNNWithMeans con similitud coseno para medir qué tan parecidos son los perfiles de valoración.

Se toma un muestra de 10.000 usuarios para reducir el uso de memoria y garantizar que el entrenamiento de K-NN sea computacionalmente viable en este entorno.

En un escenario de producción, convendría aplicar técnicas de reducción de dimensionalidad o uso de índices aproximados para escalar a todo el dataset.

In [8]:
import pandas as pd
import numpy as np

In [9]:
ratings_df = pd.read_csv("ratings_limpios.csv")
resumen_usuario = pd.read_csv("resumen_usuario.csv")

In [10]:
np.random.seed(42)

usuarios_unicos = resumen_usuario['User-ID'].unique()

# 10000 o el dataset completo si llegara a tener menos de 10000
n_muestra = min(10000, len(usuarios_unicos))

usuarios_muestra = np.random.choice(usuarios_unicos, size=n_muestra, replace=False)

sub_df = ratings_df[ratings_df['User-ID'].isin(usuarios_muestra)].copy()


In [31]:
from surprise import Dataset, Reader, KNNWithMeans
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(1,10))

surprise_df = sub_df.copy()
surprise_df.columns = ["uid", "iid", "rating"]

data = Dataset.load_from_df(surprise_df, reader)

# valores probados 0.3 y 0.4
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Entrenamiento
algo = KNNWithMeans(k=40, sim_options={'name': 'cosine', 'user_based': True}, verbose=False)
algo.fit(trainset)

predictions = algo.test(testset)

In [32]:
# Transformación a dataframe
preds_df = pd.DataFrame([
    {
        'User-ID':pred.uid,
        'ISBN': pred.iid,
        'r_ui': pred.r_ui,
        'est': pred.est
    }
    for pred in predictions
])

In [33]:
from metrics import evaluar_metricas_usuario

K = 10
metricas_usuarios = (
    preds_df
    .groupby('User-ID')
    .apply(evaluar_metricas_usuario, k=K)
    .reset_index()
)


df_evaluacion = pd.merge(
    metricas_usuarios,
    resumen_usuario,
    on='User-ID',
    how='inner',
    validate='one_to_one'
)

In [34]:
# A. Por Grupo Etario (Demográfico)
print(f"=== Métricas por Grupo Etario (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Etario')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# B. Por Historial de Interacciones (Comportamiento)
print(f"\n=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Historial')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# C. Por Grado de Exigencia (Comportamiento)
print(f"\n=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Exigencia')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

=== Métricas por Grupo Etario (K=10) ===
                   MAE      CG@10     DCG@10   NDCG@10
Grupo_Etario                                          
Adultos       1.460152  19.320189  12.961555  0.978060
Jóvenes       1.466203  17.365234  12.194837  0.981522
Mayores       1.375365  13.883117  10.957033  0.987262

=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Historial                                          
0                1.400965   7.759901   7.655392  0.999354
1                1.494049  21.121996  13.855640  0.973832

=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Exigencia                                          
0                2.572760   8.258427   6.617117  0.985851
1                1.253496  18.152751  12.514664  0.979481
2                1.887932  17.730366  13.463120  0.995108

Obserbaciones:

Grupo Etario:

Ek desempeño es bastante homogéneo. los mayores presentan el menor MAE y los góvenes el mayor, aunque las diferencias son pequñas. El NDCG es alto en los tres grupos.

Grupo separado por historial:

Los usuarios con historial corto tienen meno MAE y un NDCG casi perfecto, pero sus valores bajos de CG/DCG se deben probablemente a que tienen pocos ítems evaluados. Los usuarios con historial largo reciben mayor utilidad acumulada, aunque con un MAE ligeramente superior y menor NDCG (Seguramente cuesta encontrar vecinos que coincidan exactamente en todo su historial).

Grado de exigencia:

Los usuarios exigentes presentan claramente el mayor MAE, por lo que el modelo predice peor sus calificaciones. Los usuarios normales tienen el mejor MAE. El NDCG es alto en todos, aunque los genereosos obtienen el mejor valor.


### Test t para grupos separados por historial

In [35]:
from scipy import stats

# Separar los usuarios en dos submuestras según su historial
g_corto = df_evaluacion[df_evaluacion['Grupo_Historial'] == 0]
g_largo = df_evaluacion[df_evaluacion['Grupo_Historial'] == 1]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

print("="*60)
print(" PRUEBA T DE STUDENT (WELCH) - HISTORIAL CORTO VS. LARGO")
print("="*60)

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()
    
    # Ejecución del T-test
    t_stat, p_val = stats.ttest_ind(v_corto, v_largo, equal_var=False)
    
    # Interpretación del p-value (umbral alpha = 0.05)
    es_significativo = "SÍ (Diferencia estadísticamente significativa)" if p_val < 0.05 else "NO (Sin evidencia de diferencia)"
    
    print(f"\n[Métrica: {metrica}]")
    print(f"  • Promedio (Historial Corto) : {v_corto.mean():.4f}")
    print(f"  • Promedio (Historial Largo) : {v_largo.mean():.4f}")
    print(f"  • Estadístico t              : {t_stat:.4f}")
    print(f"  • p-value                    : {p_val:.4e}")
    print(f"  • ¿Existe inequidad?         : {es_significativo}")


 PRUEBA T DE STUDENT (WELCH) - HISTORIAL CORTO VS. LARGO

[Métrica: MAE]
  • Promedio (Historial Corto) : 1.4010
  • Promedio (Historial Largo) : 1.4940
  • Estadístico t              : -2.5253
  • p-value                    : 1.1633e-02
  • ¿Existe inequidad?         : SÍ (Diferencia estadísticamente significativa)

[Métrica: CG@10]
  • Promedio (Historial Corto) : 7.7599
  • Promedio (Historial Largo) : 21.1220
  • Estadístico t              : -31.8930
  • p-value                    : 1.7252e-191
  • ¿Existe inequidad?         : SÍ (Diferencia estadísticamente significativa)

[Métrica: DCG@10]
  • Promedio (Historial Corto) : 7.6554
  • Promedio (Historial Largo) : 13.8556
  • Estadístico t              : -33.6411
  • p-value                    : 2.6455e-213
  • ¿Existe inequidad?         : SÍ (Diferencia estadísticamente significativa)

[Métrica: NDCG@10]
  • Promedio (Historial Corto) : 0.9994
  • Promedio (Historial Largo) : 0.9738
  • Estadístico t              : 26.0287
  • p-va

La prueba T confirma diferencias significativas en las métricas evaluadas entre el historial corto vs largo.

En cuanto a la diferencia del MAE entre ambos se puede relacionar con la dificultad de encontrar vecinos que se acoplen adecuadamente con los usuarios con historial largo.

Esto puede potenciarse con la toma de la muestra donde se reduce más la cantidad de representantes de cada grupo.

### Anova para grupos etarios

In [36]:
from scipy import stats

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

print("="*60)
print(" TEST ANOVA: GRUPOS ETARIOS")
print("="*60)

for metrica in metricas:
    # 1. Agrupar las muestras por cada rango etario
    grupos = [
        datos[metrica].dropna() 
        for _, datos in df_evaluacion.groupby('Grupo_Etario')
    ]
    
    # 2. ANOVA de un factor
    f_stat, p_val = stats.f_oneway(*grupos)
    
    es_significativo = p_val < 0.05
    conclusion = "SÍ (Diferencias significativas entre edades)" if es_significativo else "NO (Métrica homogénea entre edades)"
    
    print(f"\n[Métrica: {metrica}]")
    print(f"  • Estadístico F    : {f_stat:.4f}")
    print(f"  • p-value          : {p_val:.4e}")
    print(f"  • ¿Existe inequidad?: {conclusion}")

 TEST ANOVA: GRUPOS ETARIOS

[Métrica: MAE]
  • Estadístico F    : 0.2731
  • p-value          : 7.6102e-01
  • ¿Existe inequidad?: NO (Métrica homogénea entre edades)

[Métrica: CG@10]
  • Estadístico F    : 4.0768
  • p-value          : 1.7076e-02
  • ¿Existe inequidad?: SÍ (Diferencias significativas entre edades)

[Métrica: DCG@10]
  • Estadístico F    : 3.1824
  • p-value          : 4.1655e-02
  • ¿Existe inequidad?: SÍ (Diferencias significativas entre edades)

[Métrica: NDCG@10]
  • Estadístico F    : 2.4191
  • p-value          : 8.9213e-02
  • ¿Existe inequidad?: NO (Métrica homogénea entre edades)


No se obeservan diferencias estadísticamente significativas en el error absoluto medio ni en la calidad del ranking normalizado.

El modelo asigna recomendaciones son un margen de precisión y orden similares entre los grupos de edades.

Con diferencias significativas en la ganancias acumulada mayor en el grupo de adultos.

### Anova para grupos separados por exigencia 

In [37]:
from scipy import stats

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

print("="*60)
print(" TEST ANOVA - GRADO DE EXIGENCIA")
print("="*60)

for metrica in metricas:
    # 1. Agrupar muestras por grupo de exigencia (0: Exigente, 1: Normal, 2: Generoso)
    grupos = [
        datos[metrica].dropna() 
        for _, datos in df_evaluacion.groupby('Grupo_Exigencia')
    ]
    
    # 2. ANOVA de un factor
    f_stat, p_val = stats.f_oneway(*grupos)
    
    es_significativo = p_val < 0.05
    conclusion = "SÍ (Diferencias significativas según exigencia)" if es_significativo else "NO (Métrica homogénea)"
    
    print(f"\n[Métrica: {metrica}]")
    print(f"  • Estadístico F    : {f_stat:.4f}")
    print(f"  • p-value          : {p_val:.4e}")
    print(f"  • ¿Existe inequidad?: {conclusion}")

 TEST ANOVA - GRADO DE EXIGENCIA

[Métrica: MAE]
  • Estadístico F    : 433.7044
  • p-value          : 7.3804e-171
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)

[Métrica: CG@10]
  • Estadístico F    : 55.1158
  • p-value          : 2.4796e-24
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)

[Métrica: DCG@10]
  • Estadístico F    : 115.6830
  • p-value          : 1.5430e-49
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)

[Métrica: NDCG@10]
  • Estadístico F    : 24.5684
  • p-value          : 2.4917e-11
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)


Se observan diferencias significativas en el desempeño según el grado de exigencia. La mayor disparidad aprece en MAE, donde los usuario exigentes presentan mayor error.



# Conclusión general de kNN usuario - usuario

Logra un buen desempeño general de recomendaciones, con valores altos de NDCG en los grupos. Esto indica que las recomendaciones mantienen una correspondencia razonable con las preferencias de los usuarios.

Los resultados deben interpretarse considerando el tamaño de la muestra y la cantidad de interacciones disponibles por grupo.

